# L04 MP2 Milstone 3 results

In [1]:
import time, statistics, psutil
import numpy as np
from multiprocessing import Pool

from mandelbrot import mandelbrot_serial, _worker

# --- MP2 M3: benchmark (in __main__ block) ---
N, max_iter = 1024, 100
X_MIN, X_MAX, Y_MIN, Y_MAX = -2.0, 1.0, -1.5, 1.5
# Serial baseline (Numba already warm after M1 warm-up)
_ = mandelbrot_serial(N, X_MIN, X_MAX, Y_MIN, Y_MAX, max_iter) # warmup run
times = []
for _ in range(3):
    t0 = time.perf_counter()
    mandelbrot_serial(N, X_MIN, X_MAX, Y_MIN, Y_MAX, max_iter)
    times.append(time.perf_counter() - t0)
t_serial = statistics.median(times)
for n_workers in range(1, psutil.cpu_count(logical=False) + 1):
    chunk_size = max(1, N // n_workers)
    chunks, row = [], 0
    while row < N:
        end = min(row + chunk_size, N)
        chunks.append((row, end, N, X_MIN, X_MAX, Y_MIN, Y_MAX, max_iter))
        row = end
    with Pool(processes=n_workers) as pool:
        pool.map(_worker, chunks)  # warm-up: Numba JIT in all workers
        times = []
        for _ in range(3):
            t0 = time.perf_counter()
            np.vstack(pool.map(_worker, chunks))
            times.append(time.perf_counter() - t0)
    t_par = statistics.median(times)
    speedup = t_serial / t_par
    print(f"{n_workers:2d} workers: {t_par:.3f}s, "
        f"speedup={speedup:.2f}x, eff={speedup/n_workers*100:.0f}%")



 1 workers: 0.125s, speedup=0.95x, eff=95%
 2 workers: 0.074s, speedup=1.59x, eff=80%
